In [ ]:
import numpy as np
from numpy import pi as π
from tqdm.notebook import tqdm, trange
import matplotlib.pyplot as plt
import firedrake
from firedrake import Constant, exp, sqrt, inner, grad, div, dx as dζ, ds, dS
import irksome
from irksome import Dt

In [ ]:
nz = 16
mesh = firedrake.UnitIntervalMesh(nz)

In [ ]:
family = "DG"  # <- revise later
degree = 0
density_element = firedrake.FiniteElement(family, "interval", degree)
Q = firedrake.FunctionSpace(mesh, density_element)

thickness_element = firedrake.FiniteElement("R", "interval", 0)
R = firedrake.FunctionSpace(mesh, thickness_element)

velocity_element = firedrake.FiniteElement("CG", "interval", degree + 1)
V = firedrake.VectorFunctionSpace(mesh, velocity_element)

Z = R * Q

See equation 6 in the CFM paper.
I'm using values just for phase 1 of densification to test things.

In [ ]:
a_s = Constant(0.3)       # m / yr
ρ_s = Constant(350.0)     # kg / m^3
ρ_i = Constant(917.0)     # kg / m^3
T = Constant(243.0)       # °K

def rate_constant(T):
    R = Constant(8.314)   # kJ / (mol °K)
    Q = Constant(10.16)   # kJ / mol
    c_0 = Constant(11.0)  # 1 / length
    return c_0 * exp(-Q / (R * T))

In [ ]:
z = firedrake.Function(Z)
z.sub(0).assign(0.1)
z.sub(1).assign(ρ_s)

h, ρ = firedrake.split(z)

In [ ]:
ζ, = firedrake.SpatialCoordinate(mesh)

ω_s = -a_s / h
r = ρ_s / ρ_i

c = rate_constant(T)
ω_expr = ω_s * (r + (1 - r) * exp(c * h * (1 - ζ) / r))

ω = firedrake.Function(V)
ω.interpolate(firedrake.as_vector((ω_expr,)));

In [ ]:
fig, ax = plt.subplots()
firedrake.plot(firedrake.Function(Q).interpolate(ω[0]), axes=ax);

In [ ]:
r, q = firedrake.TestFunctions(Z)

h, ρ = firedrake.split(z)

a_c = Constant(2 * a_s)
ρ_c = Constant(550.0)
a_b = a_c * (ρ - ρ_c) / (ρ_i - ρ_c)

ρ_b = Constant((ρ_c + sqrt(ρ_c**2 + 4 * (ρ_i - ρ_c) * ρ_s * a_s / a_c)) / 2)
z.sub(1).interpolate((1 - ζ) * ρ_b + ζ * ρ_s)

F_h = (
    (Dt(h) - a_s + a_b) * r * dζ +
    h * ω[0] * r * ds((1,)) - 
    h * ω[0] * r * ds((2,))
)

ν = firedrake.FacetNormal(mesh)
ω_ν = firedrake.max_value(0, inner(ω, ν))

F_ρ = (
    (Dt(h * ρ) * q - inner(h * ρ * ω, grad(q))) * dζ +
    firedrake.avg(h) * (ρ("+") * ω_ν("+") * q("+") + ρ("-") * ω_ν("-") * q("-")) * dS +
    ρ * a_b * q * ds((1,)) -
    ρ_s * a_s * q * ds((2,))
)

F = F_h + F_ρ

In [ ]:
method = irksome.BackwardEuler()
timestep = 1 / 3600
dt = Constant(timestep)
t = Constant(0.0)
solver = irksome.TimeStepper(F, method, t, dt, z)

In [ ]:
final_time = 5.0
num_steps = int(final_time / timestep)
zs = [z.copy(deepcopy=True)]
for step in trange(num_steps):
    solver.advance()
    t.assign(t + dt)
    zs.append(z.copy(deepcopy=True))

In [ ]:
fig, ax = plt.subplots()

ρ = z.subfunctions[1]
firedrake.plot(zs[2].subfunctions[1], axes=ax);